# 1. Information about the submission

## 1.1 Name and number of the assignment

Assignment 1: Word Sense Induction

## 1.2 Student name

Egor Serov

## 1.3 Codalab user ID / nickname / username

iwanttodie

## 1.4 Additional comments

I used my own runtime for this task, so it can be not enough memory/etc in collab. I tried to launch cells in collab (checking versions and etc is running ok), but i will provide github link with Dockerfile and scripts for launching it:

https://github.com/awesomeslayer/IntroToNLP2025

# 2. Technical Report

*Use Section 2 to describe results of your experiments as you would do writing a paper about your results. DO NOT insert code in this part. Only insert plots and tables summarizing results as needed. Use formulas if needed do described your methodology. The code is provided in Section 3.*

## 2.1 Methodology


### 1. General Framework
We approached the Word Sense Induction task as a semantic clustering problem within a high-dimensional vector space. Our pipeline follows a "Fine-tune & Cluster" paradigm:
1.  **Input Formatting:** augument text to focus attention on the target lexical unit.
2.  **Contrastive Fine-Tuning:** specialize a pre-trained Transformer to pull contexts with identical senses closer.
3.  **Agglomerative Clustering:** recover the latent sense inventory using a variance-minimizing hierarchical approach.

### 2. Model Architecture
We selected encoder models based on the linguistic complexity of each dataset (and avaliability of good models on 2025 year):

*   **Wiki-Wiki (Coarse-grained):** We employed `sentence-transformers/paraphrase-multilingual-mpnet-base-v2`. Since Wikipedia senses often represent distinct concepts (homonyms, e.g., *bank* as an institution vs. *bank* as land). A multilingual base model provided sufficient semantic separation with high computational efficiency. But we tried ruRoberta too but it really overfits and creates unsifficient results on test.
*   **BTS-RNC & Active-Dict (Fine-grained):** These datasets come from explanatory dictionaries involving subtle polysemy. To capture these nuances: `ai-forever/ruRoberta-large`. This 24-layer model, pre-trained on massive Russian corpora.It has necessary capacity to distinguish between shades of meaning (e.g., *light* as illumination vs. *light* as a color shade).

### 3. Contrastive Fine-Tuning
Standard pre-trained models produce embeddings where spatial proximity reflects topical similarity, not necessarily sense identity. To reshape the vector space, we fine-tuned the models using **Contrastive Learning**.

#### Input Representation
we prepended the input prompt to bias self-attention towards the target word, so this target lemma:
$$ \text{Input} = "[CLS] \text{ } <\text{word}> \text{ } ; \text{ } <\text{context}> \text{ } [SEP]" $$
This allows the $[CLS]$ token (it is for pooling) to aggregate context specifically relative to the our target word.

#### Training Objective
We used **Multiple Negatives Ranking Loss (MNRL)**. Given a batch of $B$ pairs of inputs $(x_i, y_i)$ where both $x_i$ and $y_i$ share the same gold sense ID, the loss maximizes the similarity between the positive pair while minimizing similarity to all other $B-1$ "negative" samples in the batch:

$$ \mathcal{L} = - \frac{1}{B} \sum_{i=1}^B \log \frac{e^{\text{sim}(h(x_i), h(y_i)) \cdot \sigma}}{\sum_{j=1}^B e^{\text{sim}(h(x_i), h(y_j)) \cdot \sigma}} $$

Where $h(\cdot)$ is the model encoder, $\text{sim}$ is cosine similarity, and $\sigma$ is a scaling factor. We set $\sigma \in [20, 30]$, which is significantly higher than standard values. This "temperature scaling" forces the model to create sharper decision boundaries between clusters, that really helps us dealing with such specific datasets.

#### Dynamic Pair Generation
To handle data imbalance (some senses have 2 examples, others 50), we implemented an adaptive sampling strategy:
1.  **Chain Sampling:** For large clusters, we formed pairs $(x_i, x_{i+1})$ to maintain connectivity without combinatorial explosion.
2.  **Full Mesh (All-to-All):** For small clusters ($N < 8$), typical in BTS-RNC, we generated all possible pairs within the sense group. This maximized the gradient signal extracted from limited data.

### 4. Clustering Algorithm
After training, we extracted embeddings for the test set and applied $L_2$ normalization. Then project them onto a unit hypersphere. We then performed **Agglomerative Hierarchical Clustering**.

#### Linkage Criterion: Ward
We consistently chose **Ward’s Linkage** over Average or Complete linkage. While SentenceTransformers optimize cosine similarity, Ward's method minimizes the increase in within-cluster variance (Sum of Squared Errors):

$$ \Delta(A, B) = \frac{|A| \cdot |B|}{|A| + |B|} || \vec{\mu}_A - \vec{\mu}_B ||^2 $$

**Justification:** Our experiments showed that fine-tuned embeddings form dense, spherical specifical "blobs" in the latent space. Ward's method is geometrically optimized to recover such compact structures, whereas Average linkage proved unstable, often chaining disparate outliers together (lets name it the "chaining effect").

#### Thresholding
The number of clusters $K$ is unknown. We determined $K$ dynamically by cutting the dendrogram at a distance threshold $\tau$. The optimal $\tau$ was found via Grid Search on the training set, maximizing the Adjusted Rand Index (ARI).

## 2.2 Discussion of results

### 1. Experimental Results
The table below summarizes the performance of our pipeline across the three datasets. We report the best (ARI) achieved during training, the optimal clustering threshold ($\tau$) found via grid search, and the final scores obtained on the CodaLab leaderboard for both Practice and Test phases.

| Dataset       | Model Architecture           | Train ARI (Best) | Threshold ($\tau$) | CodaLab Score (Practice) | CodaLab Score (Test) |
|:--------------|:-----------------------------|:-----------------|:-------------------|:-------------------------|:---------------------|
| **wiki-wiki** | `paraphrase-mpnet-base-v2`   | 1.0000           | 3.3322             | **0.9649**               | *[Pending]*          |
| **bts-rnc**   | `ruRoberta-large`            | 0.8628           | 1.7114             | **0.6615**               | **0.5928**           |
| **active-dict**| `ruRoberta-large`           | 0.6599           | 0.7651             | **0.5289**               | **0.4290**           |

### 2. Analysis

#### 1. The Impact of Model Capacity (Wiki vs. Dictionaries)
The results clearly delineate two levels of task difficulty.
*   **Wiki-Wiki:** We achieved near-perfect results (Practice ARI 0.965) using a standard base model. This confirms that Wikipedia senses are typically **homonyms**. The high distance threshold ($\tau \approx 3.33$) indicates that the clusters are naturally far apart in the vector space, making them easy to separate even without a large model.
*   **Dictionary Datasets (BTS & Active-Dict):** These tasks proved significantly harder due to **polysemy**. Our initial experiments with base models failed to generalize (Test ARI < 0.2). However, after switching to `ruRoberta-large` combined with aggressive fine-tuning (with really high scale=25) provided a massive improvement. It pushed the BTS test score to **0.5928**. This demonstrates that distinguishing fine linguistic nuances requires the deep semantic knowledge embedded in large, language-specific models.

#### 2. Threshold Sensitivity and Semantic Density
An interesting observation is the difference in optimal thresholds between the two dictionary datasets:
*   **BTS-RNC ($\tau \approx 1.71$):** The senses here are distinct enough to allow a moderate threshold.
*   **Active-Dict ($\tau \approx 0.77$):** The significantly lower threshold suggests that the senses in the "Active Dictionary" are **semantically denser**. The embeddings for different senses lie very close to each other, requiring a much stricter cutting to preventing merging distinct meanings into a single cluster. This aligns with the lower overall scores (Test ARI 0.429), reflecting the higher complexity of this dataset.

#### 3. Generalization Gap
We observe a natural drop between Train ARI and Test ARI (e.g., BTS: $0.86 \to 0.59$). This is expected in WSI tasks where the test set contains new contexts not seen during fine-tuning. However, the fact that the Test scores remain high confirms that the model has learned **generalized semantic representations** in a good way. Moreover, it is better that simply memorizing the training examples. The use of `Ward` linkage was crucial here, as it enforces compact, spherical clusters that generalize better than the irregular shapes often produced by Average linkage.

### 3. Visual Analysis
The plots below (generated during the training phase) illustrate the Grid Search process for the distance threshold. We selected $\tau$ at the peak of the ARI curve to maximize cluster purity.

<table>
  <tr>
    <td align="center">
      <b>Wiki-Wiki</b><br>
      <img src="reports/figures/wiki-wiki_threshold_search.png" width="400"/>
    </td>
    <td align="center">
      <b>BTS-RNC</b><br>
      <img src="reports/figures/bts-rnc_threshold_search.png" width="400"/>
    </td>
  </tr>
  <tr>
    <td align="center" colspan="2">
      <b>Active-Dict</b><br>
      <img src="reports/figures/active-dict_threshold_search.png" width="400"/>
    </td>
  </tr>
</table>
<p align="center"><em>Figure 1-3: Threshold Optimization curves across datasets.</em></p>

### 4. Conclusion
The "Fine-tune & Cluster" approach proved highly effective. For coarse-grained tasks (Wiki), standard multilingual models are sufficient. For fine-grained Russian polysemy (BTS/Active-Dict), success depends on three pillars: **Large Language Models (ruRoberta-large)**, **Contrastive Loss with high scaling**, and **Ward's Hierarchical Clustering**.

# 3. Code

*Enter here all code used to produce your results submitted to Codalab. Add some comments and subsections to navigate though your solution.*

*In this part you are expected to develop yourself a solution of the task and provide a reproducible code:*
- *Using Python 3;*
- *Contains code for installation of all dependencies;*
- *Contains code for downloading of all the datasets used*;
- *Contains the code for reproducing your results (in other words, if a tester downloads your notebook she should be able to run cell-by-cell the code and obtain your experimental results as described in the methodology section)*.


*As a result, you code will be graded according to these criteria:*
- ***Readability**: your code should be well-structured preferably with indicated parts of your approach (Preprocessing, Model training, Evaluation, etc.).*
- ***Reproducibility**: your code should be reproduced without any mistakes with “Run all” mode (obtaining experimental part).*


## 3.1 Requirements

In [ ]:
import sys
!{sys.executable} -m pip install --upgrade pip setuptools wheel
!{sys.executable} -m pip install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu118
!{sys.executable} -m pip install --upgrade numpy pandas scikit-learn matplotlib scipy tqdm transformers sentence-transformers datasets accelerate --prefer-binary
!{sys.executable} -m pip install spacy razdel nltk #building wheel will last for ~30mins do not worry) 
!{sys.executable} -m spacy download ru_core_news_sm
!pip install tabulate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 13.5 MB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 84.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 37.2 MB/s eta 0:00:00
  Attempting uninstall: wheel
    Found existing installation: wheel 0.37.1
    Uninstalling wheel-0.37.1:
      Successfully uninstalled wheel-0.37.1
  Attempting uninstall: setuptools
    Found existing installation: setuptools 65.6.3
    Uninstalling setuptools-65.6.3:
      Successfully uninstalled setuptools-65.6.3
  Attempting uninstall: pip
    Found existing installation: pip 22.3.1
    Uninstalling pip-22.3.1:
      Successfully uninstalled pip-22.3.1
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 13.8 MB/s  0:00:00
INFO: pip is looking at multiple versions of networkx to determine which version is compatible with other requirements. This

## 3.2 Download the data

In [ ]:
import os
import random
import zipfile
import warnings
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import normalize
from tqdm.auto import tqdm
import sentence_transformers
import sentence_transformers.fit_mixin
from datasets import Dataset, DatasetDict, IterableDataset
from sentence_transformers import SentenceTransformer, InputExample, losses, models

sentence_transformers.fit_mixin.Dataset = Dataset
sentence_transformers.fit_mixin.DatasetDict = DatasetDict
sentence_transformers.fit_mixin.IterableDataset = IterableDataset
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings('ignore')

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs("reports/figures", exist_ok=True)
os.makedirs("reports/tables", exist_ok=True)
os.makedirs("reports/models", exist_ok=True)

if not os.path.exists("russe-wsi-kit"):
    print("Cloning RUSSE-2018 repository...")
    !git clone https://github.com/nlpub/russe-wsi-kit.git
else:
    print("Repository already exists.")

BASE_PATH = "russe-wsi-kit/data/main"

EXPERIMENT_LOGS = []

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

set_seed(42)
print(f"Setup complete. Device: {DEVICE}")

Repository already exists.
Setup complete. Device: cuda


## 3.3 Preprocessing

In [ ]:
def format_input_text(row):
    word = str(row['word'])
    context = " ".join(str(row['context']).split())
    return f"{word} ; {context}"

def generate_pairs_smart(df, ds_name="dataset"):
    examples = []
    for word, group in df.groupby('word'):
        if 'gold_sense_id' not in group.columns: continue
        
        for sense_id, cluster in group.groupby('gold_sense_id'):
            texts = cluster.apply(format_input_text, axis=1).tolist()
            if len(texts) < 2: continue
            
            for i in range(len(texts) - 1):
                examples.append(InputExample(texts=[texts[i], texts[i+1]]))
            
            if len(texts) <= 8:
                for i in range(len(texts)):
                    for j in range(i + 1, len(texts)):
                        examples.append(InputExample(texts=[texts[i], texts[j]]))
            else:
                for _ in range(min(len(texts) * 2, 50)):
                    a, b = random.sample(texts, 2)
                    if a != b:
                        examples.append(InputExample(texts=[a, b]))
    
    print(f"  [{ds_name}] Generated {len(examples)} training pairs.")
    return examples

def train_pipeline(train_examples, model_name, batch_size, epochs, scale, ds_name, lr = 1e-5):
    print(f"Initializing model: {model_name}")
    word_embedding_model = models.Transformer(model_name, max_seq_length=128)
    pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension())
    model = SentenceTransformer(modules=[word_embedding_model, pooling_model])
    model.to(DEVICE)
    
    loader = DataLoader(train_examples, shuffle=True, batch_size=batch_size)
    loss = losses.MultipleNegativesRankingLoss(model, scale=scale)
    
    print(f"Training {ds_name}: {epochs} epochs, BS={batch_size}, Scale={scale}")
    model.fit(
        train_objectives=[(loader, loss)],
        epochs=epochs,
        warmup_steps=int(len(loader) * 0.1),
        optimizer_params={'lr': lr},
        show_progress_bar=True
    )
    return model

def optimize_threshold_and_plot(model, df_train, task_name, linkage='ward', metric='euclidean', th_range=np.linspace(1.0, 5.0, 100)):
    print(f"Optimizing Threshold for {task_name}...")
    
    texts = df_train.apply(format_input_text, axis=1).tolist()
    embeddings = normalize(model.encode(texts, batch_size=32, show_progress_bar=True))
    
    best_ari = -1
    best_th = 0
    history_th = []
    history_ari = []

    for th in th_range:
        total_ari = 0
        valid_words = 0
        
        for word, group in df_train.groupby('word'):
            idxs = group.index.to_numpy()
            if len(idxs) < 2: continue
            
            cl = AgglomerativeClustering(n_clusters=None, distance_threshold=th, linkage=linkage, metric=metric)
            try:
                preds = cl.fit_predict(embeddings[idxs])
                total_ari += adjusted_rand_score(group['gold_sense_id'], preds)
                valid_words += 1
            except: pass
            
        avg_ari = total_ari / valid_words if valid_words > 0 else 0
        history_th.append(th)
        history_ari.append(avg_ari)
        
        if avg_ari > best_ari:
            best_ari = avg_ari
            best_th = th

    plt.figure(figsize=(10, 6))
    plt.plot(history_th, history_ari, label=f'{linkage} linkage', color='b', linewidth=2)
    plt.axvline(x=best_th, color='r', linestyle='--', label=f'Best Th={best_th:.2f}')
    plt.title(f"Hyperparameter Search: Threshold vs ARI ({task_name})")
    plt.xlabel("Distance Threshold")
    plt.ylabel("Adjusted Rand Index (Train)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(f"reports/figures/{task_name}_threshold_search.png", dpi=300)
    plt.close()
    
    print(f"{task_name} Best ARI: {best_ari:.4f} at Threshold: {best_th:.4f}")
    return best_th, best_ari

def generate_submission(model, task_name, threshold, linkage='ward', metric='euclidean'):
    test_path = os.path.join(BASE_PATH, task_name, "test.csv")
    if not os.path.exists(test_path): 
        return
    
    print(f"Generating submission for {task_name}...")
    df_test = pd.read_csv(test_path, sep='\t')
    texts = df_test.apply(format_input_text, axis=1).tolist()
    emb = normalize(model.encode(texts, batch_size=32, show_progress_bar=True))
    
    preds = np.zeros(len(df_test), dtype=int)

    for word, group in df_test.groupby('word'):
        idxs = group.index.to_numpy()
        if len(idxs) < 2:
            preds[idxs] = 1
            continue
        
        cl = AgglomerativeClustering(
            n_clusters=None, 
            distance_threshold=threshold,
            linkage=linkage,
            metric=metric
        )
        preds[idxs] = cl.fit_predict(emb[idxs])
    
    df_test['predict_sense_id'] = preds

    csv_path = os.path.join(RESULTS_DIR, f"{task_name}.csv")
    zip_path = os.path.join(RESULTS_DIR, f"submission_{task_name}.zip")

    df_test.to_csv(csv_path, sep='\t', index=False)

    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.write(csv_path, arcname=f"{task_name}.csv")

    print(f"Saved: {csv_path}")
    print(f"Saved: {zip_path}")


## 3.4 My method of text processing

In [ ]:
# ========================
# TASK: WIKI-WIKI
# ========================
TASK = "wiki-wiki"
MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
PARAMS = {"epochs": 4, "scale": 20.0, "batch_size": 64, "oversample": 5}

df_wiki = pd.read_csv(os.path.join(BASE_PATH, TASK, "train.csv"), sep='\t')
examples = generate_pairs_smart(df_wiki, TASK)
train_data = examples * PARAMS["oversample"] 
random.shuffle(train_data)

model_wiki = train_pipeline(train_data, MODEL, PARAMS["batch_size"], PARAMS["epochs"], PARAMS["scale"], TASK, lr=2e-5)

best_th, best_ari = optimize_threshold_and_plot(
    model_wiki, df_wiki, TASK, 
    linkage='ward', 
    th_range=np.linspace(1.5, 5.0, 150)
)


EXPERIMENT_LOGS.append({
    "Dataset": TASK,
    "Model": MODEL,
    "Epochs": PARAMS["epochs"],
    "Scale": PARAMS["scale"],
    "Train ARI": best_ari,
    "Best Threshold": best_th
})
generate_submission(model_wiki, TASK, best_th)

  [wiki-wiki] Generated 809 training pairs.
Initializing model: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Training wiki-wiki: 4 epochs, BS=64, Scale=20.0


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


Optimizing Threshold for wiki-wiki...


Batches:   0%|          | 0/14 [00:00<?, ?it/s]

wiki-wiki Best ARI: 1.0000 at Threshold: 2.9799
Generating submission for wiki-wiki...


Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Saved: results/wiki-wiki.csv
Saved: results/submission_wiki-wiki.zip


In [ ]:
# ========================
# TASK: BTS-RNC
# ========================
torch.cuda.empty_cache()
TASK = "bts-rnc"
MODEL = "ai-forever/ruRoberta-large"
PARAMS = {"epochs": 4, "scale": 25.0, "batch_size": 16} # Large model config

df_bts = pd.read_csv(os.path.join(BASE_PATH, TASK, "train.csv"), sep='\t')
examples = generate_pairs_smart(df_bts, TASK)
random.shuffle(examples)

model_bts = train_pipeline(examples, MODEL, PARAMS["batch_size"], PARAMS["epochs"], PARAMS["scale"], TASK)

best_th, best_ari = optimize_threshold_and_plot(
    model_bts, df_bts, TASK, 
    linkage='ward', 
    th_range=np.linspace(1.5, 6.0, 150)
)

EXPERIMENT_LOGS.append({
    "Dataset": TASK,
    "Model": MODEL,
    "Epochs": PARAMS["epochs"],
    "Scale": PARAMS["scale"],
    "Train ARI": best_ari,
    "Best Threshold": best_th
})

generate_submission(model_bts, TASK, best_th, linkage='ward')

  [bts-rnc] Generated 6346 training pairs.
Initializing model: ai-forever/ruRoberta-large


Some weights of RobertaModel were not initialized from the model checkpoint at ai-forever/ruRoberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training bts-rnc: 4 epochs, BS=16, Scale=25.0


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,0.389400
1000,0.220700
1500,0.198200


Optimizing Threshold for bts-rnc...


Batches:   0%|          | 0/110 [00:00<?, ?it/s]

bts-rnc Best ARI: 0.8628 at Threshold: 1.7114
Generating submission for bts-rnc...


Batches:   0%|          | 0/205 [00:00<?, ?it/s]

Saved: results/bts-rnc.csv
Saved: results/submission_bts-rnc.zip


In [ ]:
# ========================
# TASK: ACTIVE-DICT
# ========================
torch.cuda.empty_cache()
TASK = "active-dict"
MODEL = "ai-forever/ruRoberta-large"
PARAMS = {"epochs": 4, "scale": 25.0, "batch_size": 16}

df_ad = pd.read_csv(os.path.join(BASE_PATH, TASK, "train.csv"), sep='\t')
examples = generate_pairs_smart(df_ad, TASK)
random.shuffle(examples)

model_ad = train_pipeline(examples, MODEL, PARAMS["batch_size"], PARAMS["epochs"], PARAMS["scale"], TASK)

best_th, best_ari = optimize_threshold_and_plot(
    model_ad, df_ad, TASK, 
    linkage='ward', 
    th_range=np.linspace(0.1, 7.0, 250)
)

EXPERIMENT_LOGS.append({
    "Dataset": TASK,
    "Model": MODEL,
    "Epochs": PARAMS["epochs"],
    "Scale": PARAMS["scale"],
    "Train ARI": best_ari,
    "Best Threshold": best_th
})

generate_submission(model_ad, TASK, best_th, linkage='ward')

  [active-dict] Generated 6916 training pairs.
Initializing model: ai-forever/ruRoberta-large


Some weights of RobertaModel were not initialized from the model checkpoint at ai-forever/ruRoberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training active-dict: 4 epochs, BS=16, Scale=25.0


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,0.209400
1000,0.088600
1500,0.078800


Optimizing Threshold for active-dict...


Batches:   0%|          | 0/65 [00:00<?, ?it/s]

active-dict Best ARI: 0.6599 at Threshold: 0.7651
Generating submission for active-dict...


Batches:   0%|          | 0/117 [00:00<?, ?it/s]

Saved: results/active-dict.csv
Saved: results/submission_active-dict.zip


In [ ]:
df_results = pd.DataFrame(EXPERIMENT_LOGS)
df_results.to_csv("reports/tables/experiment_summary.csv", index=False)

print("\nFINAL EXPERIMENT SUMMARY")
print(df_results.to_markdown(index=False))

print("\nAll tasks completed. Artifacts stored in 'reports/'.")
print("   - Plots: reports/figures/")
print("   - CSVs: reports/tables/")


FINAL EXPERIMENT SUMMARY
| Dataset     | Model                                                       |   Epochs |   Scale |   Train ARI |   Best Threshold |
|:------------|:------------------------------------------------------------|---------:|--------:|------------:|-----------------:|
| wiki-wiki   | sentence-transformers/paraphrase-multilingual-mpnet-base-v2 |        4 |      20 |    1        |          2.97987 |
| bts-rnc     | ai-forever/ruRoberta-large                                  |        4 |      25 |    0.862756 |          1.71141 |
| active-dict | ai-forever/ruRoberta-large                                  |        4 |      25 |    0.659915 |          0.76506 |

All tasks completed. Artifacts stored in 'reports/'.
   - Plots: reports/figures/
   - CSVs: reports/tables/
